In [ ]:
import os
# Prevenção de vazamento de memória do MKL no Windows
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

In [ ]:
# Carregar os dados
df = pd.read_csv('dados_soja.csv')

In [ ]:
# Ajustes nos dados
df['valor'] = df['valor'].replace('-', '0')              # Substitui '-' por '0'
df['valor'] = df['valor'].astype(str).str.replace('.', '', regex=False)  # Remove pontos
df['valor'] = df['valor'].astype(float)                 # Converte para float

In [ ]:
# Selecionar variável para clusterização
X = df[['valor']].values

In [ ]:
# Padronização (opcional, mas recomendado)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Testar diferentes números de clusters usando coeficiente de silhueta
silhouette_scores = []
K = range(2, 11)  # testa de 2 a 10 clusters

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}, Coeficiente de Silhueta = {score:.4f}")

# Plotando gráfico da silhueta
plt.figure(figsize=(8, 5))
plt.plot(K, silhouette_scores, marker='o')
plt.xlabel("Número de clusters (k)")
plt.ylabel("Coeficiente de Silhueta")
plt.title("Coeficiente de Silhueta para diferentes valores de k")
plt.show()

In [ ]:
# Escolher o melhor k (maior silhueta)
best_k = K[silhouette_scores.index(max(silhouette_scores))]
print(f"\nNúmero ótimo de clusters: {best_k}")

In [ ]:
# Rodar KMeans final com número ótimo de clusters
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

In [ ]:
# Visualização final
plt.figure(figsize=(10,6))
plt.scatter(df['id'], df['valor'], c=df['Cluster'], cmap='viridis')
plt.xticks(rotation=90)  # rotaciona os nomes dos estados para caber no gráfico
plt.xlabel('Estado')
plt.ylabel('Produção de Soja')
plt.title(f'Clusters de Produção de Soja por Estado (k={best_k})')
plt.colorbar(label='Cluster')
plt.show()

print(df[['id', 'valor', 'Cluster']])